In [ ]:
import numpy as np
import pandas as pd
import math

import pickle
import os

import matplotlib.pyplot as plt
import seaborn as sns


## Loading data

In [ ]:
text_data = pd.read_csv('../data/processed/stories.csv')
text_data.tail(2)

story_idx = text_data['story_id'].values


In [ ]:
text_data.agg({
        'word_count': ['count','min','max','mean','median'],
        'wc_diff': ['min','max','mean','median'],
    }) \
    .round(2)


In [ ]:
print(f"Total word count: {text_data.loc[:,['word_count']].sum().values[0]}")

text_data \
    .groupby('model') \
    .agg({
        'word_count': ['count','min','max','mean','median'],
        'wc_diff': ['min','max','mean','median'],
    }) \
    .loc[['Human', 'GPT-2', 'GPT-2 (tag)', 'Qwen2.5-7B', 'Falcon3-7B', 'gemma-2'],:] \
    .round(2)


In [ ]:
# Published raw metrics
metrics_data = pd.read_csv('../data/metrics/metrics_raw.csv')
metrics_data = metrics_data[['story_id', 'SMOG', 'ARI', 'LFP', 'MDD', 'SRP', 'SPD', 'SCC', 'SVL', 'EPD', 'ECC', 'EVL']]


In [ ]:
# Published aggregated human ratings
scores_data = pd.read_csv('../data/ratings/final_scores.csv')
scores_data = scores_data[['story_id', 'RE', 'CH', 'EM', 'SU', 'EG', 'CX', 'model']]
model_idx = scores_data[['story_id', 'model']]


## RQ1: Correlation and comparison

### Correlation

In [ ]:
scores_data


In [ ]:
cor_metrics = metrics_data.loc[:, metrics_data.columns[1:]] \
    .corr() \
    .round(4)

plt.figure(figsize=(8,8))

cor_lower = np.tril(cor_metrics.values, -1)
cor_lower[cor_lower == 0] = np.nan

plt.imshow(cor_lower, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(location='right', shrink=.8)

for i in range(cor_metrics.shape[0]):
    for j in range(cor_metrics.shape[1]):
        if j < i and abs(cor_metrics.values[i,j]) > .05:
            if abs(cor_metrics.values[i,j]) > .3:
                plt.text(
                    j, i, s=round(cor_metrics.values[i,j], 2),
                    ha='center', va='center',
                    fontsize=12, fontweight='bold'
                )
            else:
                plt.text(
                    j, i, s=round(cor_metrics.values[i,j], 2),
                    ha='center', va='center',
                    fontsize=12
                )

plt.xticks(range(metrics_data.shape[1]-1), metrics_data.columns[1:], fontsize=12, rotation=45)
plt.yticks(range(metrics_data.shape[1]-1), metrics_data.columns[1:], fontsize=12)

plt.show()


In [ ]:
all_cors = scores_data.iloc[:,:7] \
    .merge(metrics_data, how = 'left') \
    .iloc[:,1:] \
    .corr(method = 'spearman')

cor_scores = all_cors.loc[scores_data.columns[1:7], scores_data.columns[1:7]] \
    .round(4)

plt.figure(figsize=(5,5))

cor_lower = np.tril(cor_scores.values, -1)

cor_lower[cor_lower == 0] = np.nan

plt.imshow(cor_lower, cmap='RdBu_r', vmin=-1, vmax=1)
plt.colorbar(location='right', shrink=.8)

for i in range(cor_scores.shape[0]):
    for j in range(cor_scores.shape[1]):
        if j < i and abs(cor_scores.values[i,j]) > .05:
            # if abs(cor_scores.values[i,j]) > .3:
            #     plt.text(
            #         j, i, s=round(cor_scores.values[i,j], 2),
            #         ha='center', va='center',
            #         fontsize=12, fontweight='bold'
            #     )
            # else:
                plt.text(
                    j, i, s=round(cor_scores.values[i,j], 2),
                    ha='center', va='center',
                    fontsize=12
                )

plt.xticks(range(scores_data.shape[1]-2), scores_data.columns[1:7], fontsize=12, rotation=45)
plt.yticks(range(scores_data.shape[1]-2), scores_data.columns[1:7], fontsize=12)

plt.show()


In [ ]:
all_cors = scores_data.iloc[:,:7] \
        .merge(metrics_data, how = 'left') \
        .iloc[:,1:] \
        .corr(method = 'spearman') \
        .loc[scores_data.columns[1:7], metrics_data.columns[1:]] \
        .round(4)

all_cors


In [ ]:
def viz_cor_scores_metrics(scores_data, metrics_data, cor_method='spearman', title=''):
    all_cors = scores_data.iloc[:,:7] \
        .merge(metrics_data, how = 'left') \
        .iloc[:,1:] \
        .corr(method = cor_method)

    cor_scores_metrics = all_cors.loc[scores_data.columns[1:7], metrics_data.columns[1:]] \
        .round(4)

    plt.figure(figsize=(8,8))

    plt.imshow(cor_scores_metrics.values, cmap='RdBu_r', vmin = -1, vmax = 1)
    plt.colorbar(location='right', shrink = .5)

    top_indices_per_row = []
    for i in range(cor_scores_metrics.shape[0]):
        row_values = cor_scores_metrics.values[i, :]
        abs_values = np.abs(row_values)
        top3 = np.argsort(abs_values)[-3:]
        top_indices_per_row.append(set(top3))

    for i in range(cor_scores_metrics.shape[0]):
        for j in range(cor_scores_metrics.shape[1]):
            if abs(cor_scores_metrics.values[i,j]) > .05:
                if j in top_indices_per_row[i]:
                    plt.text(
                        j, i, s=round(cor_scores_metrics.values[i,j], 2),
                        ha='center', va='center',
                        fontsize=12, fontweight='bold'
                    )
                else:
                    plt.text(
                        j, i, s=round(cor_scores_metrics.values[i,j], 2),
                        ha='center', va='center',
                        fontsize=12
                    )

    plt.vlines(x=4.5, ymin=-.5, ymax=5.5, color='black', linewidth=1.2)
    plt.vlines(x=1.5, ymin=-.5, ymax=5.5, color='black', linewidth=1, linestyle='dashed')
    plt.vlines(x=7.5, ymin=-.5, ymax=5.5, color='black', linewidth=1, linestyle='dashed')

    plt.xticks(range(metrics_data.shape[1]-1), metrics_data.columns[1:], fontsize = 12, rotation = 45)
    plt.yticks(range(scores_data.shape[1]-2), scores_data.columns[1:7], fontsize = 12)

    if title:
        plt.title(title)

    plt.show()

viz_cor_scores_metrics(scores_data, metrics_data)


In [ ]:
viz_cor_scores_metrics(
    scores_data[scores_data['story_id']<1100],
    metrics_data[metrics_data['story_id']<1100]
    )

viz_cor_scores_metrics(
    scores_data[scores_data['story_id']>=1100],
    metrics_data[metrics_data['story_id']>=1100]
    )

viz_cor_scores_metrics(
    scores_data[scores_data['story_id']>100],
    metrics_data[metrics_data['story_id']>100]
    )


In [ ]:
model_label = np.unique(scores_data['model'].values)

for m in model_label:
    select_idx = scores_data[scores_data['model']==m]['story_id'].values
    viz_cor_scores_metrics(
        scores_data[scores_data['story_id'].isin(select_idx)],
        metrics_data[metrics_data['story_id'].isin(select_idx)],
        title = f'Correlartion of {m} (N = {len(select_idx)})'
        )


### Comparison

In [ ]:
text_metrics_data = text_data[['story_id', 'prompt_id', 'model']] \
    .merge(metrics_data, how='left')

text_metrics_data \
    .groupby('model') \
    .aggregate(['mean', 'std']) \
    .iloc[:,2*3:] \
    .round(2) \
    .reset_index()


In [ ]:
text_scores_data = text_data[['story_id', 'prompt_id', 'model']] \
    .merge(scores_data, how='left')

text_scores_data \
    .groupby('model') \
    .aggregate(['mean', 'std']) \
    .iloc[:,2*3:] \
    .round(2) \
    .reset_index()


In [ ]:
from scipy.stats import f_oneway
from scipy.stats import ttest_rel,ttest_ind

def get_bar_comp(df, color_palette='icefire'):
    sns_colors = sns.color_palette(color_palette, n_colors=3)
    
    def eta_squared(*groups):
        # Perform one-way ANOVA
        f_statistic, p_value = f_oneway(*groups)

        # Compute the ANOVA table values
        total_mean = np.mean(np.concatenate(groups))
        num_groups = len(groups)
        num_obs = sum(len(group) for group in groups)
        ss_total = np.sum((np.concatenate(groups) - total_mean) ** 2)
        ss_between = sum(len(group) * (np.mean(group) - total_mean) ** 2 for group in groups)
        ss_within = sum((len(group) - 1) * np.var(group, ddof=1) for group in groups)

        eta_squared = ss_between / (ss_between + ss_within)

        return f_statistic,p_value,eta_squared
    
    def cohens_d(array1, array2):
        try:
            t_stat, p_value = ttest_rel(array1, array2)
        except:
            t_stat, p_value = ttest_ind(array1, array2)

        dof = len(array1) - 1
        pooled_std = np.sqrt(((len(array1) - 1) * np.var(array1, ddof=1) + (len(array2) - 1) * np.var(array2, ddof=1)) / dof)
        cohens_d = (np.mean(array1) - np.mean(array2)) / pooled_std
        return t_stat, p_value, cohens_d

    def get_significance_stars(p_value):
        if p_value < 0.001:
            return '***'
        elif p_value < 0.01:
            return '**'
        elif p_value < 0.05:
            return '*'
        else:
            return ''

    df_long = pd.melt(df, id_vars=['model'], var_name='metric', value_name='value')
    metrics = df_long['metric'].unique()
    n_metrics = len(metrics)

    n_row = 2
    n_col = 3
    fig, axes = plt.subplots(n_row, n_col, figsize=(15, 10))

    def get_color_for_model(model_name):
        if model_name == 'Human':
            return sns_colors[1]
        elif 'gpt-2' in model_name.lower():
            return sns_colors[0]
        else:
            return sns_colors[2]

    for ax, metric in zip(axes.flat, metrics):
        metric_data = df_long[(df_long['metric'] == metric)]
        
        groups = metric_data['model'].unique()
        means = []
        stds = []
        
        all_groups_data = []
        group_names = []
        
        human_data = metric_data[metric_data['model'] == 'Human']['value'].values
        
        for group in groups:
            group_data = metric_data[metric_data['model'] == group]['value']
            means.append(group_data.mean())
            stds.append(group_data.std())
            all_groups_data.append(group_data.values)
            group_names.append(group)
        
        x_pos = np.arange(len(groups))
        colors = [get_color_for_model(group) for group in groups]
        bars = ax.bar(x_pos, means, yerr=stds, capsize=3, alpha=.8, color=colors)
        
        max_value_with_text = max(mean + std for mean, std in zip(means, stds))
        y_max = max_value_with_text * 1.25  
        ax.set_ylim(0, y_max)
        
        f_stat, p_value, eta_sq = eta_squared(*all_groups_data)
        
        if p_value < 0.001:
            p_text = '$p$ < .001'
            p_sign = '(***'
        elif p_value < 0.01:
            p_text = '$p$ < .01'
            p_sign = '(**'
        elif p_value < 0.05:
            p_text = '$p$ < .05'
            p_sign = '(*)'
        else:
            p_text = f'$p$ = {p_value:.3f}'
            p_sign = ''
        
        if eta_sq > .14:
            eta_sign = ', +)'
        elif eta_sq > .06:
            eta_sign = ', -)'
        else:
            eta_sign = ''
        
        ax.set_title(f'{metric} {p_sign}{eta_sign}\n($F$ = {f_stat:.2f}, $\\eta^2$ = {eta_sq:.2f})', 
                    fontsize=18, pad=10)
        
        for i, (mean_val, std_val) in enumerate(zip(means, stds)):
            text_y = mean_val
            ax.text(x_pos[i], text_y, 
                    f'{mean_val:.2f}\n({std_val:.2f})', 
                    ha='center', va='top', fontsize=14,
                    bbox=dict(facecolor='white', edgecolor='white', alpha=0.4, pad=1)
            )
            
            if groups[i] != 'Human':
                current_data = metric_data[metric_data['model'] == groups[i]]['value'].values
                
                t_stat, p_value, effect_size = cohens_d(human_data, current_data)
                significance = get_significance_stars(p_value)
                
                sig_y = mean_val
                ax.text(x_pos[i], sig_y, 
                        significance, 
                        ha='center', va='bottom', fontsize=25, 
                        color='black' if p_value < 0.05 else 'white',
                        bbox=dict(facecolor='white', edgecolor='white', alpha=0.4, pad=1)
                    )
                
        
        ax.hlines(y=means[0], xmin=-.5, xmax=len(groups)-0.5, color='black', linestyles='dashed', label='Human')
        
        ax.set_xticks(x_pos)
        ax.set_xticklabels(groups, fontsize=16, rotation=45, ha='right')

    for i in range(len(metrics), n_row*n_col):
        axes.flat[i].set_visible(False)

    plt.tight_layout()
    plt.show()

get_bar_comp(text_metrics_data[['model'] + text_metrics_data.columns[3:8].tolist()], color_palette='rocket')
get_bar_comp(text_metrics_data[['model'] + text_metrics_data.columns[8:].tolist()], color_palette='rocket')
get_bar_comp(text_scores_data.iloc[:,2:], color_palette='rocket')


## RQ2A: Classification and feature importance

### Mixed-effect logistic regression

See `mixed-effect-r-0330.ipynb`.

In [ ]:
text_scores_data['label'] = text_scores_data['model'].apply(
    lambda x: 1 if x == 'Human' else 0
    )
text_metrics_data['label'] = text_metrics_data['model'].apply(
    lambda x: 1 if x == 'Human' else 0
    )

from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

pre_scores_data = scaler.fit_transform(text_scores_data.iloc[:,3:-1])
pre_scores_data = pd.DataFrame(pre_scores_data, columns = text_scores_data.iloc[:,3:-1].columns)
pre_scores_data[['story_id', 'prompt_id', 'model', 'label']] = text_scores_data[['story_id', 'prompt_id', 'model', 'label']]

pre_metrics_data = scaler.fit_transform(text_metrics_data.iloc[:,3:-1])
pre_metrics_data = pd.DataFrame(pre_metrics_data, columns = text_metrics_data.iloc[:,3:-1].columns)
pre_metrics_data[['story_id', 'prompt_id', 'model', 'label']] = text_metrics_data[['story_id', 'prompt_id', 'model', 'label']]


In [ ]:
pre_scores_data.head()


In [ ]:
pre_metrics_data.head()


In [ ]:
# pre_scores_data.to_csv('pre-scores-data.csv')
# pre_metrics_data.to_csv('pre-metrics-data.csv')


In [ ]:
import statsmodels.api as sm
from statsmodels.genmod.bayes_mixed_glm import BinomialBayesMixedGLM

random_effects = {
    'model': '0 + C(model)',
    'prompt': '0 + C(prompt_id)'
}

scores_model = BinomialBayesMixedGLM.from_formula(
    f"label ~ {'+'.join(pre_scores_data.iloc[:,:-4].columns)}",   
    random_effects,
    data=pre_scores_data,        
)

scores_result = scores_model.fit_vb()    
scores_result.summary()


In [ ]:
metrics_model = BinomialBayesMixedGLM.from_formula(
    f"label ~ {'+'.join(pre_metrics_data.iloc[:,:-4].columns)}",   
    random_effects,
    data=pre_metrics_data,        
)

metrics_result = metrics_model.fit_vb()    
metrics_result.summary()


### Logistic model

In [ ]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# To avoid multicollinearity, we removed SMOG and EVL.

select_cols = ['ARI', 'LFP', 'MDD', 'SRP', 'SPD', 'SCC', 'SVL', 'EPD', 'ECC', 'story_id', 'prompt_id', 'model', 'label']
ppre_metrics_data = pre_metrics_data[select_cols]
ppre_metrics_data.head()


In [ ]:
scores_vif = pd.DataFrame({
    'index': pre_scores_data.iloc[:,:-4].columns,
    'vif': [variance_inflation_factor(pre_scores_data.iloc[:,:-4].values, i) for i in range(len(pre_scores_data.iloc[:,:-4].columns))]
}).round(2)
scores_vif


In [ ]:
metric_vif = pd.DataFrame({
    'index': ppre_metrics_data.iloc[:,:-4].columns,
    'vif': [variance_inflation_factor(ppre_metrics_data.iloc[:,:-4].values, i) for i in range(len(ppre_metrics_data.iloc[:,:-4].columns))]
}).round(2)

# metric_vif_data = pre_metrics_data[['ARI', 'LFP', 'MDD', 'SRP', 'SPD', 'SCC', 'SVL', 'EPD', 'ECC']]
# metric_vif = pd.DataFrame({
#     'index': metric_vif_data.columns,
#     'vif': [variance_inflation_factor(metric_vif_data.values, i) for i in range(len(metric_vif_data.columns))]
# }).round(2)

metric_vif


In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix, classification_report)

def get_logit_model(preprocessed_data, test_size=.2, random_iter = 1000, random_state = 42, fit_l1_wt=0, fit_alpha=0.1, return_margeff=False):
    X = preprocessed_data.iloc[:,:-4]
    y = preprocessed_data.iloc[:,-1]

    # Split dataset
    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=test_size,
        random_state=random_state,
        stratify=y          
    )
    print(f"Training set size: {X_train.shape}")
    print(f"Testing set size: {X_test.shape}")

    # Weight
    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    total = len(y_train)
    weight_neg = total / (2 * n_neg)
    weight_pos = total / (2 * n_pos)
    sample_weights = np.where(y_train == 0, weight_neg, weight_pos)

    # Logit model (ridge regression)
    logit_model = sm.Logit(y_train, X_train, weights=sample_weights)
    result = logit_model.fit_regularized(L1_wt=fit_l1_wt, alpha=fit_alpha)

    params = result.params
    bse = result.bse
    pvalues = result.pvalues
    conf_int = result.conf_int()
    marg_eff = result.get_margeff(at='mean')

    coef_df = pd.DataFrame({
        'coef': params,
        'std_err': bse,
        'z': params / bse,      
        'p>|z|': pvalues,
        'ci_lower': conf_int[0],
        'ci_upper': conf_int[1]
    })

    # Prediction accuracy
    y_pred_proba = result.predict(X_test)   
    y_pred = (y_pred_proba >= 0.5).astype(int)

    # Random baseline
    n_test = len(y_test)
    p_pos = y_train.mean()

    random_accs = []
    random_aucs = []
    for _ in range(random_iter):
        random_pred = np.random.binomial(1, p_pos, n_test)
        random_proba = np.random.uniform(0, 1, n_test) 
        random_accs.append(accuracy_score(y_test, random_pred))
        random_aucs.append(roc_auc_score(y_test, random_proba))

    random_baseline = {
        'mean_acc': np.mean(random_accs),
        'std_acc': np.std(random_accs),
        'mean_auc': np.mean(random_aucs),
        'std_auc': np.std(random_aucs)
    }

    evaluation_results = dict({
        'acc': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'auc': roc_auc_score(y_test, y_pred_proba),
        'random_baseline': random_baseline
    })

    if return_margeff:
        return coef_df, evaluation_results, marg_eff.summary_frame()
    else:
        return coef_df, evaluation_results

get_logit_model(pre_scores_data, .2)


In [ ]:
from sklearn.model_selection import StratifiedKFold

def kfold_logit_model(preprocessed_data, k=5, test_size=0.2, random_iter = 1000, random_state=42, fit_l1_wt=0, fit_alpha=0.1):
    X = preprocessed_data.iloc[:, :-4]
    y = preprocessed_data.iloc[:, -1]

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, random_state=random_state, stratify=y
    )
    print(f"Training set size: {X_train.shape}")
    print(f"Testing set size: {X_test.shape}")

    kfold = StratifiedKFold(n_splits=k, shuffle=True, random_state=random_state)

    cv_metrics = {
        'accuracy': [], 'precision': [], 'recall': [], 'f1': [], 'auc': []
    }

    for fold_idx, (train_idx, val_idx) in enumerate(kfold.split(X_train, y_train)):
        X_tr = X_train.iloc[train_idx]
        y_tr = y_train.iloc[train_idx]
        X_val = X_train.iloc[val_idx]
        y_val = y_train.iloc[val_idx]

        n_neg = (y_tr == 0).sum()
        n_pos = (y_tr == 1).sum()
        total = len(y_tr)
        weight_neg = total / (2 * n_neg) if n_neg > 0 else 1.0
        weight_pos = total / (2 * n_pos) if n_pos > 0 else 1.0
        sample_weights = np.where(y_tr == 0, weight_neg, weight_pos)

        model = sm.Logit(y_tr, X_tr, weights=sample_weights)
        result = model.fit_regularized(L1_wt=fit_l1_wt, alpha=fit_alpha)

        y_pred_proba = result.predict(X_val)
        y_pred = (y_pred_proba >= 0.5).astype(int)

        cv_metrics['accuracy'].append(accuracy_score(y_val, y_pred))
        cv_metrics['precision'].append(precision_score(y_val, y_pred, zero_division=0))
        cv_metrics['recall'].append(recall_score(y_val, y_pred, zero_division=0))
        cv_metrics['f1'].append(f1_score(y_val, y_pred, zero_division=0))
        cv_metrics['auc'].append(roc_auc_score(y_val, y_pred_proba))

    cv_summary = {}
    for metric, values in cv_metrics.items():
        cv_summary[metric] = {
            'mean': np.mean(values),
            'std': np.std(values)
        }

    print("\nEvaluation of K-fold models")
    for metric, stats in cv_summary.items():
        print(f"{metric}: mean = {stats['mean']:.4f} ± {stats['std']:.4f}")

    n_neg = (y_train == 0).sum()
    n_pos = (y_train == 1).sum()
    total = len(y_train)
    weight_neg = total / (2 * n_neg) if n_neg > 0 else 1.0
    weight_pos = total / (2 * n_pos) if n_pos > 0 else 1.0
    sample_weights = np.where(y_train == 0, weight_neg, weight_pos)

    final_model = sm.Logit(y_train, X_train, weights=sample_weights)
    final_result = final_model.fit()

    params = final_result.params
    bse = final_result.bse
    pvalues = final_result.pvalues
    conf_int = final_result.conf_int()
    coef_df = pd.DataFrame({
        'coef': params,
        'std_err': bse,
        'z': params / bse,
        'p>|z|': pvalues,
        'ci_lower': conf_int[0],
        'ci_upper': conf_int[1]
    })

    y_pred_proba_test = final_result.predict(X_test)
    y_pred_test = (y_pred_proba_test >= 0.5).astype(int)

    # Random baseline
    n_test = len(y_test)
    p_pos = y_train.mean()

    random_accs = []
    random_aucs = []
    for _ in range(random_iter):
        random_pred = np.random.binomial(1, p_pos, n_test)
        random_proba = np.random.uniform(0, 1, n_test) 
        random_accs.append(accuracy_score(y_test, random_pred))
        random_aucs.append(roc_auc_score(y_test, random_proba))

    random_baseline = {
        'mean_acc': np.mean(random_accs),
        'std_acc': np.std(random_accs),
        'mean_auc': np.mean(random_aucs),
        'std_auc': np.std(random_aucs)
    }

    test_evaluation = dict({
        'acc': accuracy_score(y_test, y_pred_test),
        'precision': precision_score(y_test, y_pred_test),
        'recall': recall_score(y_test, y_pred_test),
        'f1': f1_score(y_test, y_pred_test),
        'auc': roc_auc_score(y_test, y_pred_proba_test),
        'random_baseline': random_baseline
    })

    print("\nEvaluation of the final model")
    for metric, val in test_evaluation.items():
        if type(val) != dict:
            print(f"{metric}: {val:.4f}")
    print(random_baseline)

    return {
        'cv_results': cv_summary,
        'test_evaluation': test_evaluation,
        'final_model': final_result,
        'coef_df': coef_df
    }

results = kfold_logit_model(pre_scores_data, k=5, test_size=0.2)

results['final_model']


#### All in one

In [ ]:
get_logit_model(pre_scores_data, .3, return_margeff=True)


In [ ]:
get_logit_model(ppre_metrics_data, .3, return_margeff=True)


#### By-group 

In [ ]:
from collections import defaultdict

def get_logit_model_by_group(preprocessed_data, model_idx, test_size = .2, random_iter = 1000, random_state = 42, fit_l1_wt=0, fit_alpha=0.1, return_margeff=False):
    coef_dfs = []
    margeff_dfs = []
    evaluation_resultss = defaultdict(dict)

    for model in np.unique(model_idx['model']):
        if model != 'Human':
            coef_df, evaluation_results, marg_eff = get_logit_model(preprocessed_data[preprocessed_data['model'].isin(['Human', model])], test_size = test_size, random_iter = random_iter, random_state = random_state, fit_l1_wt=fit_l1_wt, fit_alpha=fit_alpha, return_margeff=True)

            coef_df['model'] = model
            coef_dfs.append(coef_df)
            marg_eff['model'] = model
            margeff_dfs.append(marg_eff)
            evaluation_resultss[model] = evaluation_results

    all_coef_res = pd.concat(coef_dfs).reset_index()
    all_marg_res = pd.concat(margeff_dfs).reset_index()

    if return_margeff:
        return all_coef_res, evaluation_resultss, all_marg_res
    else:
        return all_coef_res, evaluation_resultss


In [ ]:
score_coef_df, score_eval_res, score_marg_res = get_logit_model_by_group(pre_scores_data, model_idx, .3, return_margeff=True)


In [ ]:
score_coef_df.round(2) \
    .sort_values(['index', 'model'])


In [ ]:
score_marg_res.round(2) \
    .sort_values(['index', 'model'])


In [ ]:
metric_coef_df, metric_eval_res, metric_marg_res = get_logit_model_by_group(ppre_metrics_data, model_idx, .3, return_margeff=True)


In [ ]:
metric_coef_df.round(4) \
    .sort_values(['index', 'model'])


In [ ]:
metric_marg_res.round(2) \
    .sort_values(['index', 'model'])


In [ ]:
metric_eval_res


## RQ3: Model by-group

In [ ]:
score_all_coef, score_all_eval = get_logit_model(pre_scores_data, .3)
metric_all_coef, metric_all_eval = get_logit_model(ppre_metrics_data, .3)

score_all_coef = score_all_coef.reset_index()
score_all_coef['model'] = 'All'
metric_all_coef = metric_all_coef.reset_index()
metric_all_coef['model'] = 'All'


In [ ]:
def get_significance_stars(p_value):
        if p_value < 0.001:
            return '***'
        elif p_value < 0.01:
            return '**'
        elif p_value < 0.05:
            return '*'
        else:
            return ''
        
score_coef_df = pd.concat([score_coef_df, score_all_coef]).reset_index(drop=True)
metric_coef_df = pd.concat([metric_coef_df, metric_all_coef]).reset_index(drop=True)

score_coef_summ = score_coef_df \
    .pivot_table(index = 'index', columns = 'model') \
    .round(2)

metric_coef_summ = metric_coef_df \
    .pivot_table(index = 'index', columns = 'model') \
    .round(2)

# score_coef_summ.to_excel('score_coef.xlsx')
# metric_coef_summ.to_excel('metric_coef.xlsx')


In [ ]:
def get_coef_comp(df, color_palette = 'mako'):
    factors = df['index'].unique()
    n_factors = len(factors)
    n_cols = 3
    n_rows = math.ceil(n_factors / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 4 * n_rows), sharey=False)
    if n_rows == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    model_order = ['All', 'GPT-2', 'GPT-2 (tag)', 'Falcon3-7B', 'gemma-2', 'Qwen2.5-7B']
    sns_colors = sns.color_palette(color_palette, n_colors=3)  

    def get_color_for_model(model_name):
        if model_name == 'All':
            return sns_colors[1]   
        elif 'gpt-2' in model_name.lower():
            return sns_colors[0]   
        else:
            return sns_colors[2]   

    color_map = {model: get_color_for_model(model) for model in model_order}

    def get_significance_stars(p_value):
            if p_value < 0.001:
                return '***'
            elif p_value < 0.01:
                return '**'
            elif p_value < 0.05:
                return '*'
            else:
                return ''

    for i, factor in enumerate(factors):
        subset = df[df['index'] == factor].copy()
        model_order = ['All', 'GPT-2', 'GPT-2 (tag)', 'Falcon3-7B', 'gemma-2', 'Qwen2.5-7B']
        subset['model'] = pd.Categorical(subset['model'], categories=model_order, ordered=True)
        subset = subset.sort_values('model').reset_index(drop=True)
        
        bar_colors = [color_map[m] for m in subset['model']]

        ax = axes[i]
        x_pos = np.arange(len(subset))
        bars = ax.bar(x_pos, subset['coef'], yerr=subset['std_err'], capsize=3,
        color=bar_colors, alpha=.8, error_kw={'linewidth': 1.5})
        
        for j, row in subset.iterrows():
            y_pos = row['coef']
            text_va = 'top' if y_pos>=0 else 'bottom'
            signif_va = 'top' if y_pos<0 else 'bottom'

            ax.text(x_pos[j], y_pos, f"{np.round(y_pos,2)}\n({np.round(row['std_err'])})", ha='center', va=text_va, fontsize=14, bbox=dict(facecolor='white', edgecolor='white', alpha=0.4, pad=1))

            if row['p>|z|'] < 0.05:
                ax.text(x_pos[j], y_pos, get_significance_stars(row['p>|z|']), ha='center', va=signif_va, fontsize=25)

        ax.set_xticks(x_pos)
        ax.set_xticklabels(subset['model'], rotation=45, ha='right', fontsize=16)
        ax.set_ylabel('Coef.', fontsize=16)
        ax.set_title(factor, fontsize=18)
        ax.axhline(0, color='black', linestyle='--', linewidth=0.8)

    plt.tight_layout()
    plt.show()


In [ ]:
get_coef_comp(score_coef_df)


In [ ]:
get_coef_comp(metric_coef_df)


In [ ]:
score_eval_res['All'] = score_all_eval
metric_eval_res['All'] = metric_all_eval

score_eval_df = pd.DataFrame(score_eval_res).T.reset_index().iloc[:,:-1]
metric_eval_df = pd.DataFrame(metric_eval_res).T.reset_index().iloc[:,:-1]

score_eval_df


In [ ]:
metric_eval_df


## RQ4: Clustering

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA


In [ ]:
def get_lda_loading_viz(preprocessed_data, opt_n_comp = 2):
    X_scaled = preprocessed_data.iloc[:,:-4]
    y = preprocessed_data.loc[:,'model']

    # Full dimension LDA
    lda = LDA(n_components=(len(y.unique())-1))
    X_lda = lda.fit_transform(X_scaled, y)
    print(f'Explained variance ratio: {np.round(lda.explained_variance_ratio_, 3)}')

    lda = LDA(n_components=opt_n_comp)
    X_lda = lda.fit_transform(X_scaled, y)

    # Loadings
    loadings = pd.DataFrame({
        'factor': X_scaled.columns,
        'LD1': lda.scalings_[:,0],
        'LD1_rank': np.argsort(np.argsort(-np.abs(lda.scalings_[:, 0]))) + 1,
        'LD2': lda.scalings_[:,1],
        'LD2_rank': np.argsort(np.argsort(-np.abs(lda.scalings_[:, 1]))) + 1
    })

    ld1 = loadings[['LD1_rank', 'factor', 'LD1']].sort_values('LD1_rank').reset_index(drop=True)
    ld1.columns = ['Rank', 'LD1_factor', 'LD1_value']

    ld2 = loadings[['LD2_rank', 'factor', 'LD2']].sort_values('LD2_rank').reset_index(drop=True)
    ld2.columns = ['Rank', 'LD2_factor', 'LD2_value']

    ranked_loadings = pd.merge(ld1, ld2, on='Rank')

    # Visualization
    custom_order = ['Human', 'GPT-2', 'GPT-2 (tag)', 'Falcon3-7B', 'gemma-2', 'Qwen2.5-7B']

    plt.figure(figsize=(6,5))

    unique_labels = custom_order
    colors = sns.color_palette('Spectral', n_colors=len(unique_labels))

    class_centers = lda.transform(lda.means_)
    class_labels = lda.classes_

    for i, label in enumerate(unique_labels):
        plt.scatter(X_lda[y==label, 0], X_lda[y==label, 1], 
                    label=f'{label}', alpha=0.7, color=colors[i])
        
        idx = np.where(class_labels == label)[0]
        if len(idx) > 0:
            center_x = class_centers[idx, 0]
            center_y = class_centers[idx, 1]
            plt.scatter(center_x, center_y, marker='^', s=200,
                        color=colors[i], edgecolor='black', linewidth=1)
            plt.text(center_x, center_y, label, ha='center', va='center', fontsize=10)

    plt.xlabel('LD1')
    plt.ylabel('LD2')
    plt.title(f'LDA Projection\n({opt_n_comp}D, Cumul. explained variance ratio = {np.round(np.cumsum(lda.explained_variance_ratio_)[-1], 4)})')
    plt.legend()
    plt.show()

    return ranked_loadings.round(2)


In [ ]:
get_lda_loading_viz(pre_scores_data)


In [ ]:
get_lda_loading_viz(pre_metrics_data)
